In [ ]:
# ============================================
# FAST WEATHER PREDICTION USING LSTM
# Optimized for College Lab / Colab
# ============================================


# ============================================
# STEP 1: Setup Kaggle API
# ============================================

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


# ============================================
# STEP 2: Download Dataset
# ============================================

!kaggle datasets download -d muthuj7/weather-dataset


# ============================================
# STEP 3: Unzip Dataset
# ============================================

!unzip weather-dataset.zip


# ============================================
# STEP 4: Import Libraries
# ============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    LSTM,
    Dense
)


# ============================================
# STEP 5: Load Dataset
# ============================================

df = pd.read_csv("weatherHistory.csv")

print(df.head())


# ============================================
# STEP 6: Select Required Columns
# ============================================

df = df[['Formatted Date', 'Temperature (C)']]

df.columns = ['date', 'temperature']


# ============================================
# STEP 7: Reduce Dataset Size
# (FASTER TRAINING)
# ============================================

df = df.tail(3000)


# ============================================
# STEP 8: Convert Date Column
# ============================================

df['date'] = pd.to_datetime(df['date'])

df.set_index('date', inplace=True)


# ============================================
# STEP 9: Handle Missing Values
# ============================================

df.dropna(inplace=True)


# ============================================
# STEP 10: Use Temperature Data
# ============================================

data = df[['temperature']].values


# ============================================
# STEP 11: Normalize Data
# ============================================

scaler = MinMaxScaler(
    feature_range=(0,1)
)

scaled_data = scaler.fit_transform(data)


# ============================================
# STEP 12: Create Sequences
# ============================================

def create_dataset(dataset, time_step=5):

    X = []

    Y = []

    for i in range(
        len(dataset) - time_step - 1
    ):

        X.append(
            dataset[i:(i+time_step), 0]
        )

        Y.append(
            dataset[i + time_step, 0]
        )

    return np.array(X), np.array(Y)


time_step = 5

X, y = create_dataset(
    scaled_data,
    time_step
)


# ============================================
# STEP 13: Reshape for LSTM
# ============================================

X = X.reshape(
    X.shape[0],
    X.shape[1],
    1
)


# ============================================
# STEP 14: Train-Test Split
# ============================================

train_size = int(len(X) * 0.8)

X_train = X[:train_size]

X_test = X[train_size:]

y_train = y[:train_size]

y_test = y[train_size:]


# ============================================
# STEP 15: Build LSTM Model
# ============================================

model = Sequential()

model.add(

    LSTM(
        25,
        return_sequences=True,
        input_shape=(time_step,1)
    )
)

model.add(
    LSTM(25)
)

model.add(
    Dense(1)
)


# ============================================
# STEP 16: Compile Model
# ============================================

model.compile(
    loss='mean_squared_error',
    optimizer='adam'
)


# ============================================
# STEP 17: Train Model
# ============================================

model.fit(

    X_train,

    y_train,

    validation_data=(X_test, y_test),

    epochs=3,

    batch_size=32,

    verbose=1
)


# ============================================
# STEP 18: Predictions
# ============================================

train_predict = model.predict(X_train)

test_predict = model.predict(X_test)


# ============================================
# STEP 19: Inverse Transform
# ============================================

train_predict = scaler.inverse_transform(
    train_predict
)

test_predict = scaler.inverse_transform(
    test_predict
)

y_actual = scaler.inverse_transform(
    y.reshape(-1,1)
)


# ============================================
# STEP 20: Plot Results
# ============================================

plt.figure(figsize=(12,6))

plt.plot(
    df.index[:len(y_actual)],
    y_actual,
    label='Actual Temperature'
)

plt.plot(

    df.index[
        time_step:
        len(train_predict)+time_step
    ],

    train_predict,

    label='Train Prediction'
)

plt.plot(

    df.index[
        len(train_predict)+time_step:
        len(train_predict)+time_step+len(test_predict)
    ],

    test_predict,

    label='Test Prediction'
)

plt.legend()

plt.title("Fast Weather Prediction using LSTM")

plt.xlabel("Date")

plt.ylabel("Temperature")

plt.show()

: 